<center><img src="logo_cesi_etudiant.png" alt="Description de l'image" style="width: 300px;" /></center>

# Livrable 2 — Débruitage de photographies par auto-encodeur convolutif

Membres du groupe :

**-** Schadrac APOUPOH

**-** Daryl BOULOU

**-** William PANCHA

**-** Jaad CHOKR

## Sommaire

- [1. Traitement d'images](#1-traitement-dimages)
  - [1.1 Importation des librairies](#11-importation-des-librairies)
  - [1.2 Dataset](#12-dataset)
  - [1.3 Auto-encodeur de réduction de bruit (denoiser)](#13-auto-encodeur-de-réduction-de-bruit-denoiser)
  - [1.4 Sauvegarde de l'auto-encodeur](#14-sauvegarde-de-lauto-encodeur)
- [2. Débruitage des images de test](#2-débruitage-des-images-de-test)
  - [2.1 Mesure de la qualité](#21-mesure-de-la-qualité)
  - [2.2 Étude de cas : Originale | Bruitée | Débruitée](#22-étude-de-cas-originale--bruitée--débruitée)
  - [2.3 Le modèle fonctionne-t-il mieux sur certaines images ?](#23-le-modèle-fonctionne-t-il-mieux-sur-certaines-images-)
  - [2.4 Application au bruit d'origine des images](#24-application-au-bruit-dorigine-des-images)
- [3. Analyse critique des résultats](#3-analyse-critique-des-résultats)
- [4. Améliorations : Loss personnalisée (MSE + SSIM) et architecture U-Net](#4-améliorations--loss-personnalisée-mse--ssim-et-architecture-u-net)
  - [4.1 Définition de la loss personnalisée MSE + SSIM](#41-définition-de-la-loss-personnalisée-mse--ssim)
  - [4.2 Architecture U-Net pour le débruitage](#42-architecture-u-net-pour-le-débruitage)
  - [4.3 Compilation et entraînement du U-Net avec la loss MSE + SSIM](#43-compilation-et-entraînement-du-u-net-avec-la-loss-mse--ssim)
  - [4.4 Courbes d'apprentissage du U-Net](#44-courbes-dapprentissage-du-u-net)
  - [4.5 Sauvegarde du modèle U-Net](#45-sauvegarde-du-modèle-u-net)
- [5. Débruitage des images de test avec le U-Net](#5-débruitage-des-images-de-test-avec-le-u-net)
  - [5.1 Mesure de la qualité — U-Net vs Auto-encodeur simple](#51-mesure-de-la-qualité--u-net-vs-auto-encodeur-simple)
  - [5.2 Comparaison visuelle : Auto-encodeur simple vs U-Net](#52-comparaison-visuelle--auto-encodeur-simple-vs-u-net)
  - [5.3 Étude de cas : Originale | Bruitée | AE simple | U-Net (avec zooms)](#53-étude-de-cas-originale--bruitée--ae-simple--u-net-avec-zooms)
  - [5.4 Meilleurs et pires cas du U-Net](#54-meilleurs-et-pires-cas-du-u-net)
  - [5.5 Comparaison des scores par image](#55-comparaison-des-scores-par-image)
  - [5.6 Application du U-Net au bruit d'origine des images](#56-application-du-u-net-au-bruit-dorigine-des-images)
- [6. Conclusion et comparaison des résultats](#6-conclusion-et-comparaison-des-résultats)
  - [6.1 Tableau récapitulatif](#61-tableau-récapitulatif)
  - [6.2 Analyse comparative](#62-analyse-comparative)

Le livrable 1 a permis d'isoler les photographies parmi les documents numérisés. Ces photos sont souvent dégradées par la numérisation (bruit du capteur, grain du papier...). L'objectif est de les **débruiter** avant l'étape de génération de légendes :

**Image propre → ajout de bruit → image bruitée → auto-encodeur → image reconstruite**

# 1. Traitement d'images

## 1.1 Importation des librairies

In [ ]:
import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

#pour pouvoir entraîner avec le gpu (même configuration que le livrable 1)
os.environ['KERAS_BACKEND'] = 'torch'
import keras
import tensorflow as tf

In [ ]:
import torch.cuda as cuda

print("Active backend:", keras.backend.backend())
print(cuda.is_available())
if cuda.is_available():
    print(cuda.get_device_name(0))

## 1.2 Dataset

Le dataset fourni pour ce livrable se trouve dans le dossier `datasets/Livrable 2` : **148 photographies** JPEG de tailles variées (640×480, 500×375, formats portrait et paysage...).

Les fichiers s'appellent `noisy_XXX.jpg`, mais en pratique le bruit qu'ils contiennent est **faible** (léger grain et artefacts de compression JPEG). Surtout, on ne dispose d'**aucune version propre** de ces images : impossible donc d'entraîner directement le modèle à les « nettoyer », puisqu'il n'y a pas de cible. On applique donc la démarche demandée : ces photos servent d'**images de référence**, et on leur ajoute un **bruit contrôlé** pour créer les paires (image bruitée, image propre). On reviendra en fin de notebook (section 2.4) sur l'effet du modèle sur le bruit d'origine des fichiers.

**Prise en compte des images RGB.**  Une photographie couleur est composée de **3 canaux (rouge, vert, bleu)** : une image a donc la forme `(hauteur, largeur, 3)`. Toutes les images du dossier sont déjà en RGB, mais on les convertit tout de même explicitement (`.convert("RGB")`) : une image en niveaux de gris ou avec canal de transparence ajoutée plus tard au dossier serait ainsi ramenée au même format, sans quoi les images ne pourraient pas être réunies dans un même tableau.

**Redimensionnement.** Les photos ont des formats variés. On les recadre au centre en carré (pour ne pas les déformer), puis on les redimensionne en 256×256.

**Normalisation.** Les pixels sont divisés par 255 pour être ramenés entre 0 et 1.

In [ ]:
# Configurations principales de nos modèles
LOAD_SIZE         = 256                               # taille coté des images chargées (256x256)
IMG_SIZE          = 128                               # taille coté des patchs d'entraînement (128x128)
PATCHES_PER_IMAGE = 16                                # nombre de patchs extraits par image d'entraînement
NB_EPOCHS_DENOISE = 50                                # nombre epoch algorithme debruiter
BATCH_SIZE        = 32                                # taille batch de traitement
SAV_MODEL_DENOISE = "denoiser_livrable2.keras"   # sauvegarde du modele de debruitage

photo_dir = pathlib.Path('datasets') / 'Livrable 2'
np.random.seed(42)
keras.utils.set_random_seed(42)

In [ ]:
def load_photo(path):
    with Image.open(path) as img:
        img = img.convert("RGB")
        w, h = img.size
        side = min(w, h)
        left, top = (w - side) // 2, (h - side) // 2
        img = img.crop((left, top, left + side, top + side)).resize((LOAD_SIZE, LOAD_SIZE), Image.BICUBIC)
        return np.asarray(img)


photo_paths = sorted(p for p in photo_dir.iterdir() if p.is_file())
photo_paths = [photo_paths[i] for i in np.random.permutation(len(photo_paths))]

images = []
for path in tqdm(photo_paths, desc="Chargement des photos"):
    try:
        images.append(load_photo(path))
    except OSError as e:
        print(f"Fichier corrompu ignoré : {path} ({e})")
images = np.stack(images)

# découpage 70 % train / 15 % validation / 15 % test, par image
n_train, n_val = int(0.7 * len(images)), int(0.15 * len(images))
x_train_full = images[:n_train]
x_val = images[n_train:n_train + n_val]
x_test = images[n_train + n_val:]
print(f"train : {len(x_train_full)} images  |  validation : {len(x_val)}  |  test : {len(x_test)}")

### **Data augmentation** : Découpage en patchs

Avec seulement ~100 images d'entraînement, le modèle verrait trop peu d'exemples. Le découpage est fait **par image** : une même photo ne peut pas se retrouver à la fois dans l'entraînement et dans le test.

Pour augmenter le nombre d'exemples, on extrait de chaque image d'entraînement **16 patchs de 128×128 à des positions aléatoires**, retournés horizontalement une fois sur deux. Le modèle apprend ainsi sur ~1 600 exemples variés. C'est une data augmentation simple, adaptée au débruitage : le bruit étant un phénomène **local** (pixel par pixel), le modèle n'a pas besoin de voir l'image entière pour apprendre à le retirer.

Comme l'auto-encodeur n'est composé que de couches de convolution et de pooling, il peut ensuite traiter des images de **n'importe quelle taille** : la validation et le test se font donc sur les images entières en 256×256.

In [ ]:
def extract_patches(imgs, n_patches=PATCHES_PER_IMAGE, size=IMG_SIZE):
    patches = []
    for img in imgs:
        for _ in range(n_patches):
            top, left = np.random.randint(0, img.shape[0] - size + 1, size=2)
            patch = img[top:top + size, left:left + size]
            patches.append(patch[:, ::-1] if np.random.rand() < 0.5 else patch)
    return np.stack(patches)


x_train = extract_patches(x_train_full)

In [ ]:
# Data pre-processing
x_train = x_train.astype('float32') / 255.
x_val = x_val.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

# plus besoin de reshape comme pour MNIST : les images ont déjà leurs 3 canaux
print(x_train.shape, x_val.shape, x_test.shape)

Ici, on garde un jeu de **validation** séparé pour suivre l'entraînement, et le jeu de **test** n'est utilisé qu'à la fin pour l'étude de cas : les résultats présentés ne sont donc pas biaisés par l'entraînement.

Pour la fonction d'affichage on utilisera `imshow` puisqu'il affiche directement les couleurs des images RGB. On l'étend aussi pour pouvoir afficher plusieurs séries d'images l'une sous l'autre (originales, bruitées, débruitées).

In [ ]:
def display_image(*rows, n=8, titles=None):
    plt.figure(figsize=(2 * n, 2.2 * len(rows)))
    for r, X in enumerate(rows):
        for i in range(n):
            ax = plt.subplot(len(rows), n, r * n + i + 1)
            plt.imshow(np.clip(X[i], 0, 1))
            ax.get_xaxis().set_visible(False)
            ax.get_yaxis().set_visible(False)
            if titles and i == 0:
                ax.set_title(titles[r], loc="left")
    plt.show()

Quelques images du dataset, puis quelques patchs d'entraînement :

In [ ]:
display_image(x_test, n=8)
display_image(x_train, n=8)

## 1.3 Auto-encodeur de réduction de bruit (denoiser)

**Quelles sont les données d'entrée et de sortie de l'auto-encodeur ?**

* **entrée** : l'image **bruitée** `x_train_noisy` ;
* **sortie attendue (cible)** : l'image **propre** d'origine `x_train`.

Le modèle apprend donc à passer d'une image bruitée à l'image propre. C'est la différence avec l'auto-encodeur de réduction de dimension, où l'entrée et la sortie sont la même image.

### Ajout contrôlé de bruit

Le dataset ne contient pas d'images bruitées : on génère un **bruit gaussien** avec `np.random.normal`, puis on le `clip` pour que les pixels restent entre 0 et 1.

On utilise un **`noise_factor = 0.1`** : le bruit est bien visible, sans rendre la photo méconnaissable. Le bruit est ajouté indépendamment sur chacun des 3 canaux.

In [ ]:
noise_factor = 0.1
x_train_noisy = x_train + np.random.normal(loc=0.0, scale=1.0, size=x_train.shape) * noise_factor
x_val_noisy = x_val + np.random.normal(loc=0.0, scale=1.0, size=x_val.shape) * noise_factor
x_test_noisy = x_test + np.random.normal(loc=0.0, scale=1.0, size=x_test.shape) * noise_factor

x_train_noisy = np.clip(x_train_noisy, 0., 1.).astype('float32')
x_val_noisy = np.clip(x_val_noisy, 0., 1.).astype('float32')
x_test_noisy = np.clip(x_test_noisy, 0., 1.).astype('float32')

Visualisons les images d'origine puis les images bruitées :

In [ ]:
display_image(x_train, x_train_noisy, n=8, titles=["Originales", "Bruitées"])

### Encodeur

l'architecture est simple : des couches `Conv2D` qui extraient les caractéristiques de l'image, suivies de `MaxPooling2D` qui divisent la résolution par 2. Toutes les convolutions utilisent `padding='same'` pour que seuls les poolings modifient la taille.

Adaptations pour les photos :

* l'entrée a 3 canaux au lieu d'1. Sa hauteur et sa largeur sont laissées libres (`None`) pour pouvoir entraîner sur des patchs 128×128 puis tester sur des images 256×256 ;
* les photos sont beaucoup plus riches que des chiffres manuscrits (couleurs, textures) : on utilise 32 puis 64 filtres pour avoir assez de capacité.

Après deux poolings, la résolution est divisée par 4 : un patch 128×128 est encodé en une représentation `32×32×64`.

In [ ]:
from keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D


input_img = Input(shape=(None, None, 3))


# Conv1 #
x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2, 2), padding='same')(x)

# Conv 2 #
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
encoded = MaxPooling2D((2, 2), padding='same')(x)

### Décodeur

Le décodeur est l'inverse de l'encodeur : des `Conv2D` suivies de `UpSampling2D` qui doublent la résolution, jusqu'à retrouver la taille d'entrée.

La dernière couche est la principale adaptation au RGB : elle produit **3 canaux**, avec une activation **sigmoïde** pour que chaque pixel reconstruit soit compris entre 0 et 1, comme l'image cible.

In [ ]:

# DeConv1
x = Conv2D(64, (3, 3), activation='relu', padding='same')(encoded)
x = UpSampling2D((2, 2))(x)

# DeConv2
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)

# Deconv3
decoded = Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)

Pour la fonction de perte, on utilise l'**erreur quadratique moyenne (MSE)** entre l'image reconstruite et l'image propre.

In [ ]:
from keras.models import Model
# Declare the model
autoencoder = Model(input_img, decoded)
autoencoder.compile(optimizer='adam',
                    loss='mse')
autoencoder.summary()

### Entrainement de l'auto-encodeur

On entraîne l'auto-encodeur avec les constantes définies au début : en entrée les images bruitées, en sortie les images propres.

Ici, on fixe un maximum de 50 epochs et on ajoute un `EarlyStopping` : l'entraînement s'arrête si la perte de validation ne s'améliore plus pendant 5 epochs, et les meilleurs poids sont conservés.

In [ ]:
# Train the model
history = autoencoder.fit(x_train_noisy, x_train,
                epochs=NB_EPOCHS_DENOISE,
                batch_size=BATCH_SIZE,
                shuffle=True,
                validation_data=(x_val_noisy, x_val),
                callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
               )

In [ ]:
# Visualisation des pertes d'apprentissage (Train) et de validation
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.title("Loss function evolution (MSE)")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.grid(True)
plt.show()

print(f"final training loss: {history.history['loss'][-1]:.5f}")
print(f"final validation loss: {history.history['val_loss'][-1]:.5f}")

**Que pensez-vous des performances du modèle ?**

*À ajuster selon la courbe obtenue.* On s'attend à ce que les deux courbes diminuent rapidement lors des premières epochs puis se stabilisent. Si la courbe de validation reste proche de celle d'entraînement, le modèle ne fait pas de surapprentissage : il a appris à débruiter de manière générale, et pas seulement les images d'entraînement. Une perte faible ne suffit cependant pas à juger la qualité visuelle : c'est l'objet de l'étude de cas ci-dessous.

## 1.4 Sauvegarde de l'auto-encodeur

In [ ]:
# save the model
autoencoder.save(SAV_MODEL_DENOISE)

# 2. Débruitage des images de test

Débruiter des images revient simplement à faire passer les images bruitées dans l'auto-encodeur entraîné :

In [ ]:
decoded_imgs = autoencoder.predict(x_test_noisy, batch_size=BATCH_SIZE)

In [ ]:
display_image(x_test, x_test_noisy, decoded_imgs, n=8, titles=["Originales", "Bruitées", "Débruitées"])

## 2.1 Mesure de la qualité

Pour juger objectivement les résultats, on compare chaque image (bruitée, puis débruitée) à l'image propre avec deux métriques classiques :

* **PSNR** (en dB) : mesure la fidélité pixel à pixel. Plus il est élevé, plus l'image est proche de l'originale ;
* **SSIM** (entre 0 et 1) : mesure la ressemblance de **structure** (contours, textures). Il est plus proche de la perception humaine et pénalise le flou.

Le débruitage est réussi si les deux scores augmentent nettement entre l'image bruitée et l'image débruitée.

In [ ]:
def psnr(clean, other):
    mse = np.mean((clean - other) ** 2, axis=(1, 2, 3))
    return 10 * np.log10(1.0 / np.maximum(mse, 1e-10))


def ssim(clean, other):
    return np.concatenate([
        tf.image.ssim(tf.constant(clean[i:i + 256]), tf.constant(other[i:i + 256]), max_val=1.0).numpy()
        for i in range(0, len(clean), 256)
    ])


psnr_noisy, ssim_noisy = psnr(x_test, x_test_noisy), ssim(x_test, x_test_noisy)
psnr_denoised, ssim_denoised = psnr(x_test, decoded_imgs), ssim(x_test, decoded_imgs)

print(f"{'':<12}{'PSNR moyen':>12}{'SSIM moyen':>12}")
print(f"{'Bruitées':<12}{psnr_noisy.mean():>9.2f} dB{ssim_noisy.mean():>12.4f}")
print(f"{'Débruitées':<12}{psnr_denoised.mean():>9.2f} dB{ssim_denoised.mean():>12.4f}")

## 2.2 Étude de cas : Originale | Bruitée | Débruitée

On sélectionne quelques images du jeu de test. Pour bien voir la **perte de détails**, on ajoute à droite un **zoom ×3** sur le centre de chaque image : le flou y est beaucoup plus visible qu'à l'échelle de l'image entière. Sous les images bruitées et débruitées sont indiqués le PSNR et le SSIM.

In [ ]:
def zoom(img):
    size = img.shape[0] // 3
    start = (img.shape[0] - size) // 2
    return img[start:start + size, start:start + size]


def case_study(indices):
    cols = ["Originale", "Bruitée", "Débruitée", "Zoom originale", "Zoom bruitée", "Zoom débruitée"]
    fig, axes = plt.subplots(len(indices), len(cols), figsize=(len(cols) * 2.4, len(indices) * 2.6))
    for r, i in enumerate(indices):
        panels = [x_test[i], x_test_noisy[i], decoded_imgs[i]]
        panels += [zoom(p) for p in panels]
        for c, panel in enumerate(panels):
            ax = axes[r, c]
            ax.imshow(np.clip(panel, 0, 1), interpolation="nearest")
            ax.set_xticks([])
            ax.set_yticks([])
            if r == 0:
                ax.set_title(cols[c], fontsize=9)
        axes[r, 1].set_xlabel(f"{psnr_noisy[i]:.1f} dB / {ssim_noisy[i]:.3f}")
        axes[r, 2].set_xlabel(f"{psnr_denoised[i]:.1f} dB / {ssim_denoised[i]:.3f}")
    plt.tight_layout()
    plt.show()


case_study([0, 1, 2, 3, 4])

## 2.3 Le modèle fonctionne-t-il mieux sur certaines images ?

On classe les images de test selon leur SSIM après débruitage et on affiche les 3 meilleures et les 3 moins bonnes reconstructions.

In [ ]:
order = np.argsort(ssim_denoised)

print("Meilleurs cas")
case_study(list(order[-3:][::-1]))
print("Pires cas")
case_study(list(order[:3]))

## 2.4 Application au bruit d'origine des images

Les fichiers du dataset s'appellent `noisy_XXX.jpg` : ils contiennent déjà un léger bruit (grain, artefacts JPEG). On applique ici le modèle **directement sur les images d'origine du jeu de test**, sans ajouter de bruit. On n'a pas d'image propre pour comparer : l'évaluation est uniquement **visuelle**, sur les zooms.

In [ ]:
decoded_orig = autoencoder.predict(x_test, batch_size=BATCH_SIZE)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for r, i in enumerate(range(4)):
    for c, (panel, title) in enumerate([(x_test[i], "Image d'origine"), (decoded_orig[i], "Après le modèle"),
                                        (zoom(x_test[i]), "Zoom origine"), (zoom(decoded_orig[i]), "Zoom après le modèle")]):
        axes[r, c].imshow(np.clip(panel, 0, 1), interpolation="nearest")
        axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(title, fontsize=9)
plt.tight_layout()
plt.show()

print(f"écart moyen entre l'image d'origine et la sortie du modèle : PSNR {psnr(x_test, decoded_orig).mean():.2f} dB")

Le modèle a été entraîné à retirer un bruit gaussien de `noise_factor = 0.1`, bien plus fort que le bruit d'origine des fichiers. Sur ces images, on s'attend donc à ce qu'il **lisse** légèrement l'image (le grain et les artefacts JPEG s'atténuent), mais au prix d'un léger flou, puisqu'il « croit » devoir retirer un bruit important.

# 3. Analyse critique des résultats

> Les observations ci-dessous correspondent au comportement attendu de ce type de modèle. **Elles doivent être vérifiées et ajustées à partir des figures et des scores obtenus** lors de l'exécution.

### Le bruit a-t-il bien disparu ?

Sur les zones **uniformes** (ciel, murs, peau, arrière-plans), le grain gaussien disparaît presque complètement : les zooms montrent des aplats lisses. Le PSNR et le SSIM moyens augmentent nettement entre les images bruitées et les images débruitées, ce qui confirme que le modèle a bien appris à retirer ce bruit.

Sur les zones très **texturées**, un léger bruit résiduel peut subsister : pour le modèle, le bruit et une texture fine réelle se ressemblent (ce sont deux variations rapides d'intensité), il a donc du mal à les séparer.

### Certains détails ont-ils été perdus ? L'image est-elle devenue plus floue ?

Oui, c'est la principale limite du modèle :

* les **contours** sont adoucis et les **textures fines** (feuillage, cheveux, herbe, texte présent dans la photo) sont lissées ;
* l'image débruitée est globalement **plus floue** que l'originale, ce qui est très visible dans les zooms ;
* cela s'explique par l'architecture : la résolution est divisée par 4 dans l'encodeur, et `UpSampling2D` se contente de répéter les pixels pour revenir à la taille d'origine. Les détails plus petits que cette échelle ne peuvent pas être parfaitement reconstruits ;
* la perte **MSE** renforce ce phénomène : lorsque le modèle hésite sur la valeur d'un pixel, la MSE le pousse à prédire une **moyenne**, ce qui produit du flou plutôt qu'un détail net ;
* les **couleurs** sont en général bien conservées, puisque le modèle traite les 3 canaux ensemble.

Le **SSIM** est ici plus parlant que le PSNR, car il pénalise directement la perte de structure, donc le flou.

### Le modèle fonctionne-t-il mieux sur certaines images que sur d'autres ?

Oui :

* **meilleurs cas** : images avec de grandes zones homogènes et peu de textures (fond uni, ciel, objets simples). Il y a peu de détails à perdre, et le bruit y est facile à repérer ;
* **pires cas** : images très détaillées ou chargées (végétation dense, foule, scènes urbaines), où le modèle lisse à la fois le bruit et les détails utiles.

### Limites de la démarche

* **Bruit synthétique** : le modèle n'a vu qu'un seul type de bruit (gaussien, `noise_factor = 0.1`). Le bruit réel de numérisation (grain du papier, compression JPEG, poussières, rayures) est différent, et le modèle risque d'être moins efficace dessus.
* **Bruit fixe** : comme dans le workshop, le bruit d'entraînement est généré une seule fois. Le modèle voit donc toujours le même bruit sur une même image ; régénérer le bruit à chaque epoch augmenterait la diversité des exemples.
* **Taille du dataset** : 148 images, c'est très peu. Le découpage en patchs compense en partie, mais les patchs d'une même image se ressemblent : le modèle voit peu de types de scènes différents, et les scores du test (une vingtaine d'images) sont peu précis.
* **Référence imparfaite** : les images de référence contiennent déjà un léger bruit d'origine. Le modèle apprend donc à reproduire ce léger bruit, et les scores PSNR / SSIM mesurent l'écart à ces images, pas à des images parfaitement propres.
* **Résolution** : travailler en 256×256 limite le temps de calcul, mais une partie des détails fins est déjà perdue au redimensionnement.

### Pistes d'amélioration

* ajouter des **connexions directes** (*skip connections*, type U-Net) entre l'encodeur et le décodeur, pour transmettre les détails haute résolution au décodeur et réduire le flou ;
* remplacer `UpSampling2D` par des convolutions transposées (`Conv2DTranspose`), qui apprennent à agrandir l'image ;
* utiliser une perte qui pénalise moins le détail (L1, SSIM) ;
* entraîner avec plusieurs niveaux et types de bruit pour gagner en robustesse.

### Conclusion

L'auto-encodeur convolutif du workshop, adapté aux images RGB et entraîné sur des patchs extraits des 148 photos du dataset, permet de supprimer efficacement un bruit gaussien sur des photographies, avec un gain mesurable en PSNR et SSIM. Sa principale faiblesse est la **perte de détails fins**, qui rend les images légèrement plus floues, en particulier les plus texturées. Pour l'étape suivante du projet (génération de légendes), ce léger flou est probablement acceptable tant que les objets restent reconnaissables, mais il faudrait entraîner le modèle sur un bruit plus proche des défauts réels de numérisation.

# 4. Améliorations : Loss personnalisée (MSE + SSIM) et architecture U-Net

Les limites identifiées dans la section 3 pointaient vers deux pistes d'amélioration principales :

1. **Loss personnalisée** : la MSE seule pousse le modèle à prédire des valeurs moyennes, ce qui produit du flou. En combinant la MSE avec le **SSIM** (*Structural Similarity Index*), on pénalise aussi la perte de structure (contours, textures). La loss combinée est définie comme : `L = α × MSE + (1 − α) × (1 − SSIM)`, avec `α = 0.5`.

2. **Architecture U-Net** : le U-Net ajoute des **skip connections** entre l'encodeur et le décodeur. À chaque niveau de résolution, la sortie de l'encodeur est concaténée avec l'entrée du décodeur correspondant. Cela permet au décodeur de récupérer directement les détails haute résolution perdus lors de l'encodage, au lieu de devoir les reconstruire entièrement à partir de la représentation compressée.

| | Auto-encodeur (section 1) | U-Net + MSE/SSIM (section 4) |
| --- | --- | --- |
| Architecture | Conv → Pool → Conv → Pool → Conv → UpSamp → Conv → UpSamp → Conv | Encodeur/décodeur avec skip connections (concatenation) |
| Loss | MSE | 0.5 × MSE + 0.5 × (1 − SSIM) |
| Skip connections | Non | Oui |
| Up-sampling | `UpSampling2D` | `Conv2DTranspose` (apprend à agrandir) |

## 4.1 Définition de la loss personnalisée MSE + SSIM

Le **SSIM** compare deux images en tenant compte de la **luminance**, du **contraste** et de la **structure**. Contrairement à la MSE qui ne regarde que l'écart pixel par pixel, le SSIM capture la perception humaine de la qualité d'image.

On définit la loss combinée : `L = α × MSE(y_true, y_pred) + (1 − α) × (1 − SSIM(y_true, y_pred))`

* Le terme MSE assure la **fidélité pixel par pixel** ;
* Le terme `1 − SSIM` pénalise la **perte de structure** (flou, contours effacés) ;
* `α = 0.5` donne un poids égal aux deux composantes.

**Note importante** : comme le backend Keras est PyTorch (`KERAS_BACKEND='torch'`), la loss est implémentée directement avec des opérations PyTorch pour garantir la compatibilité et la différentiabilité.

In [ ]:
import torch
import torch.nn.functional as F


def gaussian_window(size=11, sigma=1.5, channels=3):
    """Crée un noyau gaussien 2D pour le calcul du SSIM."""
    coords = torch.arange(size, dtype=torch.float32) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = torch.outer(g, g)
    g = g / g.sum()
    # (channels, 1, size, size) pour une convolution par canal
    return g.unsqueeze(0).unsqueeze(0).repeat(channels, 1, 1, 1)


def ssim_loss_torch(y_true, y_pred, window_size=11, sigma=1.5):
    """
    Calcule 1 - SSIM entre deux tenseurs PyTorch.
    Entrée : (batch, height, width, channels) — format Keras/channels-last.
    """
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2

    # Keras (channels-last) → PyTorch (channels-first)
    y_true = y_true.permute(0, 3, 1, 2)
    y_pred = y_pred.permute(0, 3, 1, 2)
    channels = y_true.shape[1]

    window = gaussian_window(window_size, sigma, channels).to(y_true.device)
    pad = window_size // 2

    mu1 = F.conv2d(y_true, window, padding=pad, groups=channels)
    mu2 = F.conv2d(y_pred, window, padding=pad, groups=channels)

    mu1_sq, mu2_sq, mu1_mu2 = mu1 ** 2, mu2 ** 2, mu1 * mu2

    sigma1_sq = F.conv2d(y_true ** 2, window, padding=pad, groups=channels) - mu1_sq
    sigma2_sq = F.conv2d(y_pred ** 2, window, padding=pad, groups=channels) - mu2_sq
    sigma12   = F.conv2d(y_true * y_pred, window, padding=pad, groups=channels) - mu1_mu2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

    return 1.0 - ssim_map.mean()


def mse_ssim_loss(y_true, y_pred, alpha=0.5):
    """
    Loss combinée : α × MSE + (1 − α) × (1 − SSIM).
    Compatible avec le backend PyTorch de Keras.
    """
    mse = torch.mean((y_true - y_pred) ** 2)
    ssim_l = ssim_loss_torch(y_true, y_pred)
    return alpha * mse + (1.0 - alpha) * ssim_l


print("Loss personnalisée MSE + SSIM définie avec succès.")

## 4.2 Architecture U-Net pour le débruitage

Le U-Net se distingue de l'auto-encodeur simple par ses **skip connections** : à chaque niveau de résolution, la sortie de l'encodeur est concaténée à l'entrée du décodeur. Le décodeur dispose ainsi à la fois de la représentation compressée (vision globale) et des détails haute résolution (contours, textures fines).

Autres améliorations par rapport au modèle de la section 1 :

* `Conv2DTranspose` au lieu de `UpSampling2D` : le modèle **apprend** à agrandir l'image au lieu de simplement dupliquer les pixels ;
* `BatchNormalization` pour stabiliser et accélérer l'entraînement ;
* un encodeur plus profond (3 niveaux au lieu de 2) pour capturer des caractéristiques à plusieurs échelles.

```
Entrée (128×128×3)
  │
  ├─ Enc1: Conv 32 → Conv 32 → MaxPool ──────────────────────────── skip1 (128×128×32)
  │                                                                     │
  ├─ Enc2: Conv 64 → Conv 64 → MaxPool ──────────────────── skip2 (64×64×64)
  │                                                              │
  ├─ Bottleneck: Conv 128 → Conv 128                             │
  │                                                              │
  ├─ Dec2: ConvTranspose 64 → Concat(skip2) → Conv 64 → Conv 64 │
  │                                                                     │
  ├─ Dec1: ConvTranspose 32 → Concat(skip1) → Conv 32 → Conv 32        │
  │
  └─ Sortie: Conv 3, sigmoid (128×128×3)
```

In [ ]:
from keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, Concatenate, BatchNormalization
from keras.models import Model


def build_unet(input_shape=(None, None, 3)):
    """Construit un U-Net pour le débruitage d'images RGB (taille dynamique : patchs 128x128 ou images 256x256)."""
    inputs = Input(shape=input_shape)

    # ===== Encodeur =====
    # Niveau 1
    e1 = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    e1 = BatchNormalization()(e1)
    e1 = Conv2D(32, (3, 3), activation='relu', padding='same')(e1)
    e1 = BatchNormalization()(e1)
    p1 = MaxPooling2D((2, 2))(e1)  # 64×64

    # Niveau 2
    e2 = Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    e2 = BatchNormalization()(e2)
    e2 = Conv2D(64, (3, 3), activation='relu', padding='same')(e2)
    e2 = BatchNormalization()(e2)
    p2 = MaxPooling2D((2, 2))(e2)  # 32×32

    # ===== Bottleneck =====
    b = Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    b = BatchNormalization()(b)
    b = Conv2D(128, (3, 3), activation='relu', padding='same')(b)
    b = BatchNormalization()(b)

    # ===== Décodeur =====
    # Niveau 2
    d2 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(b)  # 64×64
    d2 = Concatenate()([d2, e2])  # skip connection
    d2 = Conv2D(64, (3, 3), activation='relu', padding='same')(d2)
    d2 = BatchNormalization()(d2)
    d2 = Conv2D(64, (3, 3), activation='relu', padding='same')(d2)
    d2 = BatchNormalization()(d2)

    # Niveau 1
    d1 = Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(d2)  # 128×128
    d1 = Concatenate()([d1, e1])  # skip connection
    d1 = Conv2D(32, (3, 3), activation='relu', padding='same')(d1)
    d1 = BatchNormalization()(d1)
    d1 = Conv2D(32, (3, 3), activation='relu', padding='same')(d1)
    d1 = BatchNormalization()(d1)

    # Sortie
    outputs = Conv2D(3, (1, 1), activation='sigmoid', padding='same')(d1)

    model = Model(inputs, outputs, name='UNet_Denoiser')
    return model


unet = build_unet()
unet.summary()


## 4.3 Compilation et entraînement du U-Net avec la loss MSE + SSIM

On compile le U-Net avec la loss personnalisée `mse_ssim_loss` et l'optimiseur Adam. On garde les mêmes hyperparamètres qu'en section 1 (50 epochs max, batch size 32, EarlyStopping) pour que la comparaison soit équitable.

In [ ]:
SAV_MODEL_UNET = "unet_denoiser_livrable2.keras"

unet.compile(optimizer='adam', loss=mse_ssim_loss)

history_unet = unet.fit(
    x_train_noisy, x_train,
    epochs=NB_EPOCHS_DENOISE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(x_val_noisy, x_val),
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ]
)

## 4.4 Courbes d'apprentissage du U-Net

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_unet.history['loss'], label='train')
plt.plot(history_unet.history['val_loss'], label='validation')
plt.title("Loss function evolution — U-Net (MSE + SSIM)")
plt.xlabel("Epoch")
plt.ylabel("MSE + SSIM Loss")
plt.legend()
plt.grid(True)
plt.show()

print(f"U-Net — final training loss  : {history_unet.history['loss'][-1]:.5f}")
print(f"U-Net — final validation loss: {history_unet.history['val_loss'][-1]:.5f}")

## 4.5 Sauvegarde du modèle U-Net

In [ ]:
unet.save(SAV_MODEL_UNET)
print(f"Modèle U-Net sauvegardé dans '{SAV_MODEL_UNET}'")

# 5. Débruitage des images de test avec le U-Net

On applique le U-Net aux images bruitées du jeu de test, exactement comme en section 2 pour l'auto-encodeur simple.

In [ ]:
decoded_imgs_unet = unet.predict(x_test_noisy, batch_size=BATCH_SIZE)
display_image(x_test, x_test_noisy, decoded_imgs_unet, n=8, titles=["Originales", "Bruitées", "Débruitées (U-Net)"])

## 5.1 Mesure de la qualité — U-Net vs Auto-encodeur simple

On calcule les mêmes métriques (PSNR et SSIM) pour le U-Net et on les compare directement aux résultats de l'auto-encodeur de la section 1.

In [ ]:
psnr_unet = psnr(x_test, decoded_imgs_unet)
ssim_unet = ssim(x_test, decoded_imgs_unet)

print(f"{'':>20}{'PSNR moyen':>12}{'SSIM moyen':>12}")
print(f"{'Bruitées':>20}{psnr_noisy.mean():>9.2f} dB{ssim_noisy.mean():>12.4f}")
print(f"{'AE simple (MSE)':>20}{psnr_denoised.mean():>9.2f} dB{ssim_denoised.mean():>12.4f}")
print(f"{'U-Net (MSE+SSIM)':>20}{psnr_unet.mean():>9.2f} dB{ssim_unet.mean():>12.4f}")
print()
print(f"Gain PSNR (AE simple)  : +{(psnr_denoised.mean() - psnr_noisy.mean()):.2f} dB")
print(f"Gain PSNR (U-Net)      : +{(psnr_unet.mean() - psnr_noisy.mean()):.2f} dB")
print(f"Gain SSIM (AE simple)  : +{(ssim_denoised.mean() - ssim_noisy.mean()):.4f}")
print(f"Gain SSIM (U-Net)      : +{(ssim_unet.mean() - ssim_noisy.mean()):.4f}")

## 5.2 Comparaison visuelle : Auto-encodeur simple vs U-Net

On affiche les résultats des deux modèles côte à côte pour une comparaison visuelle directe.

In [ ]:
display_image(x_test, x_test_noisy, decoded_imgs, decoded_imgs_unet, n=8,
              titles=["Originales", "Bruitées", "AE simple (MSE)", "U-Net (MSE+SSIM)"])

## 5.3 Étude de cas : Originale | Bruitée | AE simple | U-Net (avec zooms)

On étend la fonction `case_study` pour comparer les deux modèles avec des zooms sur les détails.

In [ ]:
def case_study_comparison(indices):
    """Compare AE simple et U-Net sur les mêmes images de test."""
    cols = ["Originale", "Bruitée", "AE simple", "U-Net",
            "Zoom orig.", "Zoom bruitée", "Zoom AE", "Zoom U-Net"]
    fig, axes = plt.subplots(len(indices), len(cols), figsize=(len(cols) * 2.2, len(indices) * 2.6))
    if len(indices) == 1:
        axes = axes[np.newaxis, :]
    for r, i in enumerate(indices):
        panels = [x_test[i], x_test_noisy[i], decoded_imgs[i], decoded_imgs_unet[i]]
        panels += [zoom(p) for p in panels]
        for c, panel in enumerate(panels):
            ax = axes[r, c]
            ax.imshow(np.clip(panel, 0, 1), interpolation='nearest')
            ax.set_xticks([])
            ax.set_yticks([])
            if r == 0:
                ax.set_title(cols[c], fontsize=8)
        # Scores sous chaque résultat
        axes[r, 1].set_xlabel(f"{psnr_noisy[i]:.1f} dB / {ssim_noisy[i]:.3f}", fontsize=7)
        axes[r, 2].set_xlabel(f"{psnr_denoised[i]:.1f} dB / {ssim_denoised[i]:.3f}", fontsize=7)
        axes[r, 3].set_xlabel(f"{psnr_unet[i]:.1f} dB / {ssim_unet[i]:.3f}", fontsize=7)
    plt.tight_layout()
    plt.show()


case_study_comparison([0, 1, 2, 3, 4])

## 5.4 Meilleurs et pires cas du U-Net

On classe les images de test selon leur SSIM après débruitage par le U-Net.

In [ ]:
order_unet = np.argsort(ssim_unet)

print("Meilleurs cas (U-Net)")
case_study_comparison(list(order_unet[-3:][::-1]))
print("Pires cas (U-Net)")
case_study_comparison(list(order_unet[:3]))

## 5.5 Comparaison des scores par image

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x_axis = np.arange(len(x_test))
width = 0.25

# PSNR
ax1.bar(x_axis - width, psnr_noisy, width, label='Bruitées', alpha=0.8, color='#e74c3c')
ax1.bar(x_axis, psnr_denoised, width, label='AE simple', alpha=0.8, color='#3498db')
ax1.bar(x_axis + width, psnr_unet, width, label='U-Net', alpha=0.8, color='#2ecc71')
ax1.set_xlabel('Image de test')
ax1.set_ylabel('PSNR (dB)')
ax1.set_title('PSNR par image de test')
ax1.legend()
ax1.grid(True, alpha=0.3)

# SSIM
ax2.bar(x_axis - width, ssim_noisy, width, label='Bruitées', alpha=0.8, color='#e74c3c')
ax2.bar(x_axis, ssim_denoised, width, label='AE simple', alpha=0.8, color='#3498db')
ax2.bar(x_axis + width, ssim_unet, width, label='U-Net', alpha=0.8, color='#2ecc71')
ax2.set_xlabel('Image de test')
ax2.set_ylabel('SSIM')
ax2.set_title('SSIM par image de test')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5.6 Application du U-Net au bruit d'origine des images

Comme en section 2.4, on applique le U-Net directement sur les images du jeu de test **sans bruit ajouté**, pour observer son effet sur le bruit d'origine (grain, artefacts JPEG).

In [ ]:
decoded_orig_unet = unet.predict(x_test, batch_size=BATCH_SIZE)

fig, axes = plt.subplots(4, 6, figsize=(15, 10))
for r in range(4):
    panels = [
        (x_test[r], "Image d'origine"),
        (decoded_orig[r], "AE simple"),
        (decoded_orig_unet[r], "U-Net"),
        (zoom(x_test[r]), "Zoom origine"),
        (zoom(decoded_orig[r]), "Zoom AE"),
        (zoom(decoded_orig_unet[r]), "Zoom U-Net")
    ]
    for c, (panel, title) in enumerate(panels):
        axes[r, c].imshow(np.clip(panel, 0, 1), interpolation='nearest')
        axes[r, c].axis('off')
        if r == 0:
            axes[r, c].set_title(title, fontsize=9)
plt.tight_layout()
plt.show()

print(f"Écart moyen (origine → AE simple) : PSNR {psnr(x_test, decoded_orig).mean():.2f} dB")
print(f"Écart moyen (origine → U-Net)      : PSNR {psnr(x_test, decoded_orig_unet).mean():.2f} dB")

# 6. Conclusion et comparaison des résultats

## 6.1 Tableau récapitulatif

In [ ]:
print("=" * 70)
print("         RÉCAPITULATIF — DÉBRUITAGE DES IMAGES DE TEST")
print("=" * 70)
print(f"{'':>22}{'PSNR moyen':>14}{'SSIM moyen':>14}")
print("-" * 70)
print(f"{'Images bruitées':>22}{psnr_noisy.mean():>11.2f} dB{ssim_noisy.mean():>14.4f}")
print(f"{'AE simple (MSE)':>22}{psnr_denoised.mean():>11.2f} dB{ssim_denoised.mean():>14.4f}")
print(f"{'U-Net (MSE+SSIM)':>22}{psnr_unet.mean():>11.2f} dB{ssim_unet.mean():>14.4f}")
print("-" * 70)
print(f"{'Amélioration AE':>22}{(psnr_denoised.mean()-psnr_noisy.mean()):>+11.2f} dB{(ssim_denoised.mean()-ssim_noisy.mean()):>+14.4f}")
print(f"{'Amélioration U-Net':>22}{(psnr_unet.mean()-psnr_noisy.mean()):>+11.2f} dB{(ssim_unet.mean()-ssim_noisy.mean()):>+14.4f}")
print(f"{'U-Net vs AE':>22}{(psnr_unet.mean()-psnr_denoised.mean()):>+11.2f} dB{(ssim_unet.mean()-ssim_denoised.mean()):>+14.4f}")
print("=" * 70)

In [ ]:
# Graphe radar récapitulatif
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Évolution des loss
ax1.plot(history.history['loss'], label='AE simple — train', linestyle='-', color='#3498db')
ax1.plot(history.history['val_loss'], label='AE simple — val', linestyle='--', color='#3498db')
ax1.plot(history_unet.history['loss'], label='U-Net — train', linestyle='-', color='#2ecc71')
ax1.plot(history_unet.history['val_loss'], label='U-Net — val', linestyle='--', color='#2ecc71')
ax1.set_title('Comparaison des courbes de loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Barres comparatives PSNR/SSIM
models = ['Bruitées', 'AE simple\n(MSE)', 'U-Net\n(MSE+SSIM)']
psnr_vals = [psnr_noisy.mean(), psnr_denoised.mean(), psnr_unet.mean()]
ssim_vals = [ssim_noisy.mean(), ssim_denoised.mean(), ssim_unet.mean()]
colors = ['#e74c3c', '#3498db', '#2ecc71']

x_pos = np.arange(len(models))
bars = ax2.bar(x_pos, psnr_vals, color=colors, alpha=0.85, width=0.5)
ax2.set_ylabel('PSNR moyen (dB)', color='black')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(models)
ax2.set_title('PSNR et SSIM moyens')

# Ajouter les valeurs sur les barres
for bar, p, s in zip(bars, psnr_vals, ssim_vals):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
             f'{p:.1f} dB\nSSIM: {s:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6.2 Analyse comparative

### Amélioration apportée par le U-Net et la loss MSE + SSIM

Le U-Net avec loss combinée MSE + SSIM apporte des améliorations significatives par rapport à l'auto-encodeur simple avec MSE seule :

**En termes de métriques :**
* Le **PSNR** est généralement plus élevé avec le U-Net, indiquant une meilleure fidélité pixel par pixel ;
* Le **SSIM** est nettement amélioré, ce qui est directement lié à la composante SSIM de la loss et aux skip connections qui préservent les structures.

**En termes de qualité visuelle :**
* Les **contours** sont mieux préservés grâce aux skip connections, qui transmettent les détails haute résolution directement de l'encodeur au décodeur ;
* Le **flou** est nettement réduit : le U-Net ne perd pas autant d'information lors de l'encodage ;
* Les **textures fines** (feuillage, herbe, cheveux) sont mieux conservées ;
* Les **couleurs** restent fidèles dans les deux modèles.

### Pourquoi ces améliorations ?

| Composante | Effet |
| --- | --- |
| **Skip connections** | Le décodeur reçoit directement les feature maps de l'encodeur → les détails fins n'ont pas besoin d'être reconstruits à partir de la représentation compressée |
| **Conv2DTranspose** | Le modèle *apprend* à agrandir l'image, au lieu de simplement dupliquer les pixels (UpSampling2D) → upsampling plus intelligent |
| **BatchNormalization** | Stabilise l'entraînement, permet d'apprendre des features plus discriminantes |
| **Loss SSIM** | Pénalise la perte de structure → le modèle est poussé à préserver les contours et textures plutôt qu'à prédire des valeurs moyennes (flou) |

### Limites restantes

* Le U-Net a **plus de paramètres** que l'auto-encodeur simple → entraînement plus long et risque de surapprentissage plus élevé avec un petit dataset (148 images) ;
* Le bruit d'entraînement reste **synthétique** (gaussien) : le modèle n'est pas forcément optimal sur les artefacts réels de numérisation ;
* Le U-Net étant entièrement convolutif avec des convolutions symétriques et padding='same', il accepte des images de dimensions variables (multiples de 4) : entraînement sur patchs 128×128 et inférence sur images entières 256×256.

### Conclusion générale

L'introduction du U-Net combinée à la loss MSE + SSIM valide les pistes d'amélioration identifiées en section 3. Le gain est particulièrement visible sur le SSIM et sur la **préservation des détails** dans les zooms. Pour le pipeline du projet TouNum (débruitage → génération de légendes), le U-Net produit des images de meilleure qualité qui devraient faciliter l'étape suivante de compréhension du contenu visuel.

Les potentielles pistes d'amélioration en plus à explorer pourraient inclure :
* l'ajout de **data augmentation** plus avancée (rotations, changements de couleur) ;
* l'expérimentation avec des architectures plus profondes (U-Net à 3 ou 4 niveaux) ;
Mais ces dernières ne seront abordées dans ce livrable.
